## Part 1: Setup and Building Data Pipeline

In [ ]:
#  import deep learning and vision libraries
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

#  enable GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('Memory growth enabled for GPU(s).')
    except RuntimeError as e:
        print(f'Memory growth cannot be set: {e}')

#  mount drive
from google.colab import drive
drive.mount('/content/drive')

#  define Data Directory
data_dir = '/content/drive/MyDrive/operation_skywatch/russian_fighterjets'

#  hyperparameters
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

#  create training and validation splits automatically
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print('Detected Classes: ', class_names)

#  optimize pipeline performance for massive datasets
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
import os

#  we are writing a for loop to read all the items in our google drive
for item_name in os.listdir(data_dir):
    print(item_name)
    #  use item_name (which is in the class name) to construct the path to the subdirectory
    for sub_item_name in os.listdir(os.path.join(data_dir, item_name)):
        print(f'   {sub_item_name}')

## Part II: Improved Convolutional Nueral Networks (CNN Model)

In [ ]:
#  1. Aggressive data augmentation to prevent overfitting on aerial angles
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2,),
    layers.RandomContrast(0.2)
])

#  2. Base model: mobileNetV2 (excellent for edge deployment on drones)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False #  freeze base model initially

#  3. Build the Classification Head
model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./127.5, offset=1),  #  normalize pixel values to [-1, 1]
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),  #  regularization to prevent overfitting
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

#  4. train the model
epochs = 25
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

## Part 3: Testing and Evaluation Code

In [ ]:
#  1. Plot training history
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.show()

plt.subplot(1, 2, 1)
plt.plot(loss, label='Training loss')
plt.plot(val_loss, label='Validation loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

#  2. Generate predictions for the validation set
print('\nEvaluating model performance...')
y_true = []
y_pred= []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

#  3. Classification report (precision, recall, f1-score)
print('\nClassification report: ')
print(classification_report(y_true, y_pred, target_names=class_names))

#  4. Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('predicted jet')
plt.ylabel('True jet')
plt.title('Confusion Matrix: Russian Fighter Jets')
plt.show()

## Testing CNN Model

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image

print('Please upload an image of a Russian fighter jet for classification')
#  this will prompt you to select a file from your computer
uploaded = files.upload()

for fn in uploaded.keys():
    #  1. get the path of the uploaded image
    path = '/content/' + fn

    #  2. load and resize the image to match the model's expected input shape (244x244)
    img = image.load_img(path, target_size=(IMG_HEIGHT, IMG_WIDTH))

    #  3. convert iamge to array and add a batch dimension *shape becomes 1 x 244 x 244 x 3
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis = 0)

    #  4. make the prediction
    predictions = model.predict(img_array)

    #  5. extract the class with the highest probability
    confidence = np.max(predictions[0]) * 100
    predicted_class_index = np.argmax(predictions[0])
    predicted_class_name = class_names[predicted_class_index]

    #  6. display the image alongside the prediction results
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Detected: {predicted_class_name}\nConfidence: {confidence:.2f}%',
              fontsize=14, fontweight='bold', color='green' if confidence > 50 else 'red')
    plt.show()

## YOLO MODEL (YOLOv8)

In [ ]:
# INstall Required libraries
!pip install -q split-folders ultralytics

import splitfolders
from ultralytics import YOLO

#  1. split data into train/val folders for YOLOv8 architecture
output_yolo_dir = '/content/yolo_dataset'
splitfolders.ratio(data_dir, output=output_yolo_dir, seed=42, ratio=(0.8, 0.2), group_prefix=None)

#  2. Initialize YOLOv8 Classification model (nano version for speed)
yolo_model = YOLO('yolov8n-cls.pt')

#  3. train YOLOv8
#  yolo automatically logs metrics, confusion matricies, and weights to the 'runs/classify/train' directory
results= yolo_model.train(
    data=output_yolo_dir,
    epochs=25,
    imgsz=224,
    batch=32,
    name='nga_skywatch_yolov8'
)

#  4. Validate the YOLO model
metrics = yolo_model.val()
print(f'YOLOv8 Top-1 accuracy: {metrics.top1:.4f}')

## Test the YOLO Model

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mping
from google.colab import files
from ultralytics import YOLO

print('Please uplead an image of a Russian fighter jet for YOLOv8 classification')
uploaded = files.upload()

#  1. locate the trained weights
#   YOLOv8 saves these automatically during the model.train() step
#  'nga_skywatch_yolov8' matches the 'name' parameter from the training code
trained_weights_path = 'runs/classify/nga_skywatch_yolov8/weights/best.pt'

#  load the trained model (or fallback to the base model if the path is disrupted)
if os.path.exists(trained_weights_path):
    inference_model = YOLO(trained_weights_path)
else:
    print(f'warning: trained weights not found at {trained_weights_path}. using model from memory')
    inference_model = yolo_model

for fn in uploaded.keys():
    #  2. get the path of the uploaded image
    img_path = '/content/' + fn

    #  3. run YOLO inference on the image
    results = inference_model.predict(img_path)

    #  4. extract the prediction details from the results object
    result = results[0]
    top_class_index = result.probs.top1
    confidence = result.probs.top1conf.item() * 100
    predicted_class_name = result.names[top_class_index]

    #  5. dispaly the image alongside the YOLO prediction results
    img = mping.imread(img_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'YOLOv8 Detected: {predicted_class_name}\nConfidence: {confidence:.2f}%',
              fontsize=14, fontweight='bold', color='green' if confidence > 50
              else 'red')
    plt.show()